In [8]:
# ============================================================
# NOTEBOOK TITLE:
# Parameter-Efficient Fine-Tuning of LLM using LoRA and PEFT
# ============================================================

# ============================================================
# 1. INTRODUCTION
# ============================================================

"""
Large Language Models (LLMs) such as GPT, LLaMA, and BERT contain billions of parameters.
Fine-tuning the entire model requires large GPU memory and computational cost.

Parameter-Efficient Fine-Tuning (PEFT) solves this by updating only a small subset
of parameters while keeping most pretrained weights frozen.

LoRA (Low-Rank Adaptation) is one of the most popular PEFT techniques.
Instead of updating full weight matrices, LoRA injects trainable low-rank matrices.

Benefits:
- Lower GPU memory usage
- Faster training
- Smaller checkpoints
- Similar performance to full fine-tuning
"""


'\nLarge Language Models (LLMs) such as GPT, LLaMA, and BERT contain billions of parameters.\nFine-tuning the entire model requires large GPU memory and computational cost.\n\nParameter-Efficient Fine-Tuning (PEFT) solves this by updating only a small subset\nof parameters while keeping most pretrained weights frozen.\n\nLoRA (Low-Rank Adaptation) is one of the most popular PEFT techniques.\nInstead of updating full weight matrices, LoRA injects trainable low-rank matrices.\n\nBenefits:\n- Lower GPU memory usage\n- Faster training\n- Smaller checkpoints\n- Similar performance to full fine-tuning\n'

In [9]:

# ============================================================
# 2. CHALLENGES OF FULL FINE-TUNING
# ============================================================

"""
Problems with full fine-tuning:

1. Huge memory requirements
2. Expensive GPU usage
3. Large storage for multiple task-specific models
4. Slow training
5. Difficult deployment

Example:
A 7B parameter model may need 28GB+ VRAM for training.
LoRA reduces trainable parameters to less than 1%.
"""


'\nProblems with full fine-tuning:\n\n1. Huge memory requirements\n2. Expensive GPU usage\n3. Large storage for multiple task-specific models\n4. Slow training\n5. Difficult deployment\n\nExample:\nA 7B parameter model may need 28GB+ VRAM for training.\nLoRA reduces trainable parameters to less than 1%.\n'

In [10]:

# ============================================================
# 3. INSTALL DEPENDENCIES
# ============================================================

!pip install transformers datasets peft accelerate evaluate -q


In [11]:

# ============================================================
# 4. IMPORT LIBRARIES
# ============================================================

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel
)

# ============================================================
# 5. DATASET PREPARATION
# ============================================================

"""
We create a small sentiment dataset.
Label:
0 -> Negative
1 -> Positive
"""

data = {
    "text": [
        "I love this product",
        "This is amazing",
        "Worst experience ever",
        "I hate this service",
        "Excellent quality",
        "Very disappointing"
    ],
    "label": [1, 1, 0, 0, 1, 0]
}

dataset = Dataset.from_dict(data)
dataset = dataset.train_test_split(test_size=0.3)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(train_dataset)
print(test_dataset)

# ============================================================
# 6. LOAD PRETRAINED MODEL
# ============================================================

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# ============================================================
# 7. TOKENIZATION
# ============================================================

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function)
test_dataset = test_dataset.map(tokenize_function)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)


Dataset({
    features: ['text', 'label'],
    num_rows: 4
})
Dataset({
    features: ['text', 'label'],
    num_rows: 2
})


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [12]:

# ============================================================
# 8. UNDERSTANDING LORA
# ============================================================

"""
LoRA decomposes weight updates into:

W + ΔW

Where:
ΔW = A × B

A and B are low-rank matrices.

Instead of training the full matrix:
Train only A and B.

Hyperparameters:
r = rank
alpha = scaling factor
dropout = regularization
"""

# ============================================================
# 9. APPLY LORA CONFIGURATION
# ============================================================

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    inference_mode=False
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# ============================================================
# 10. TRAINING CONFIGURATION
# ============================================================

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    logging_steps=1,
    save_strategy="epoch",
    evaluation_strategy="epoch"
)

# ============================================================
# 11. EVALUATION METRIC
# ============================================================

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}

# ============================================================
# 12. TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# ============================================================
# 13. TRAIN MODEL
# ============================================================

trainer.train()

# ============================================================
# 14. EVALUATION
# ============================================================

results = trainer.evaluate()
print("Evaluation Results:")
print(results)

# ============================================================
# 15. SAVE LORA ADAPTER
# ============================================================

save_path = "./lora_adapter"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Adapter saved successfully")

# ============================================================
# 16. LOAD SAVED ADAPTER
# ============================================================

base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

loaded_model = PeftModel.from_pretrained(
    base_model,
    save_path
)

print("Adapter loaded successfully")

# ============================================================
# 17. INFERENCE TEST
# ============================================================

sample_text = "This product is wonderful"

inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

with torch.no_grad():
    outputs = loaded_model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=-1)

label_map = {
    0: "Negative",
    1: "Positive"
}

print("Prediction:", label_map[prediction.item()])

# ============================================================
# 18. COMPARISON: FULL FT VS LORA
# ============================================================

comparison = {
    "Method": ["Full Fine-Tuning", "LoRA"],
    "Trainable Params": ["100%", "<1%"],
    "GPU Memory": ["Very High", "Low"],
    "Speed": ["Slow", "Fast"],
    "Storage": ["Large", "Small"]
}

for i in range(len(comparison["Method"])):
    print("Method:", comparison["Method"][i])
    print("Trainable Params:", comparison["Trainable Params"][i])
    print("GPU Memory:", comparison["GPU Memory"][i])
    print("Speed:", comparison["Speed"][i])
    print("Storage:", comparison["Storage"][i])
    print()


ValueError: Please specify `target_modules` or `target_parameters`in `peft_config`

In [ ]:

# ============================================================
# 19. OBSERVATIONS
# ============================================================

"""
Observations:

1. LoRA drastically reduces trainable parameters.
2. Training becomes memory efficient.
3. Adapter files are small compared to full model checkpoints.
4. Performance remains competitive for downstream tasks.
5. PEFT is ideal for production systems with limited resources.
"""

# ============================================================
# 20. CONCLUSION
# ============================================================

"""
This notebook demonstrated:

✔ LLM fine-tuning fundamentals
✔ Challenges of full fine-tuning
✔ LoRA concept
✔ PEFT implementation
✔ Dataset preparation
✔ Model loading
✔ Fine-tuning workflow
✔ Evaluation
✔ Saving/loading adapters

Final Conclusion:
LoRA + PEFT enables efficient adaptation of large transformer models
without retraining billions of parameters.
"""